# VideoMark: ProPainter and DiffuEraser

Repository: [https://github.com/KYRIE-LI11/VideoMark](https://github.com/KYRIE-LI11/VideoMark)

Paper: [https://arxiv.org/abs/2504.16359](https://arxiv.org/abs/2504.16359)

Inspected commit: `9f8d78b73ab9f9f055651b1b4f37d68bdb05e7be`

Watermark stage: `in_generation`
Domain: video

This method is coupled to generation, so the notebook uses only author-provided or author-generated native data. It does not fake a MOSE embedding.

The two removers receive the exact same watermarked frames and binary mask. One official DiffuEraser execution is used: its internal ProPainter prior is preserved as the ProPainter result, and its final refinement is the DiffuEraser result. Inputs shorter than 22 frames are padded by repeating only the last frame/mask to 24 for inference, then both reported outputs are trimmed to the authentic length.

This notebook uses the authors' committed `a_red_panda_eating_leaves` ModelScope watermarked MP4, committed PRC key, and committed shift value. Because VideoMark publishes no object annotation for that sample, the attack is a recorded 35% centered region mask—not an author mask.


## Authenticity contract

- Watermark preparation: Use the authors' committed ModelScope red-panda watermarked sample, shift value, and 512-bit PRC key; then create and record one aligned removal mask.
- Checkpoint status: READY FOR ATTACK: the watermarked sample, shift, and keys are committed. Native extraction still requires the upstream ModelScope inversion model.
- Protocol note: VideoMark cannot be retrofitted onto an existing MOSE clip.  Only a natively generated VideoMark output may be passed to ProPainter or DiffuEraser.
- No result cell fabricates metrics. Native extraction must finish and write its JSON before values are displayed.
- A method without a released spatial detector reports bit recovery only; it is not assigned a proxy localizer.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/remaining_watermark_methods.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"

def run(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", shlex.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def require(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path

def require_commit(repository, expected):
    actual = subprocess.check_output(["git", "-C", str(repository), "rev-parse", "HEAD"], text=True).strip()
    if actual != expected:
        raise RuntimeError(f"Expected {expected} in {repository}, found {actual}")

def show_results(run_root):
    run_root = Path(run_root)
    for name in ("watermarked.mp4", "aligned-mask.mp4"):
        path = run_root / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    attack = run_root / "attack"
    for name in ("input-watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4"):
        path = attack / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    for path in sorted(run_root.rglob("summary.json")):
        print(path.relative_to(run_root))
        display(JSON(json.loads(path.read_text())))

print("Repository:", ROOT)


In [ ]:
VIDEOMARK = Path("/path/to/VideoMark")
VIDEOMARK_PYTHON = VIDEOMARK / ".venv/bin/python"
MODELSCOPE_MODEL = Path("/path/to/damo-vilab/text-to-video-ms-1.7b")
DIFFUERASER = Path("/path/to/DiffuEraser")
DIFFUERASER_PYTHON = DIFFUERASER / ".venv/bin/python"
GPU = "0"
SAMPLE = VIDEOMARK / "result/videomark/modelscope/512bit/a_red_panda_eating_leaves/wm.mp4"
SHIFT = VIDEOMARK / "result/videomark/modelscope/512bit/a_red_panda_eating_leaves/shift_value.npy"
KEYS = VIDEOMARK / "keys/64_64_512bit.pkl"
RUN_ROOT = ROOT / "runs/remaining-watermarks/videomark/a_red_panda_eating_leaves"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WATERMARKED_VIDEO = SAMPLE
ALIGNED_MASK = RUN_ROOT / "aligned-mask.mp4"


In [ ]:
require_commit(VIDEOMARK, "9f8d78b73ab9f9f055651b1b4f37d68bdb05e7be")
for path, label in [(SAMPLE, "authors' watermarked sample"), (SHIFT, "shift value"), (KEYS, "PRC key"), (MODELSCOPE_MODEL, "ModelScope inversion model")]:
    require(path, label)


In [ ]:
run([
    DIFFUERASER_PYTHON, SCRIPTS / "make_native_removal_mask.py", "center-box",
    "--video", SAMPLE, "--output-video", ALIGNED_MASK,
    "--output-frames", RUN_ROOT / "aligned-mask-frames",
    "--width-fraction", "0.35", "--height-fraction", "0.35",
])


## Run the identical two-method attack

The preparation cell freezes hashes, frame count, dimensions, FPS, mask foreground counts, provenance, and any short-video padding. The next cell runs both methods from that frozen input.


In [ ]:
ATTACK_RUN = RUN_ROOT / "attack"
if not (ATTACK_RUN / "sample.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "prepare_watermark_removal_sample.py", "prepare",
        "--source-video", WATERMARKED_VIDEO, "--source-mask", ALIGNED_MASK,
        "--output", ATTACK_RUN, "--watermark-method", "VideoMark",
        "--sample-id", "a_red_panda_eating_leaves", "--sample-source", "VideoMark authors' committed ModelScope 512-bit sample",
        "--source-kind", "authors_provided",
    ])
else:
    print("Prepared sample already exists:", ATTACK_RUN / "sample.json")


In [ ]:
if not (ATTACK_RUN / "removal-pair.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "run_watermark_removal_pair.py",
        "--run", ATTACK_RUN, "--diffueraser", DIFFUERASER,
        "--diffueraser-python", DIFFUERASER_PYTHON,
        "--device", GPU, "--seed", "2025",
    ])
else:
    print("Both removal outputs already exist:", ATTACK_RUN / "removal-pair.json")


In [ ]:
run([
    VIDEOMARK_PYTHON, SCRIPTS / "run_videomark_detection.py",
    "--source", VIDEOMARK, "--model-path", MODELSCOPE_MODEL,
    "--keys", KEYS, "--shift", SHIFT, "--output", RUN_ROOT / "detection",
    "--case", f"original={ATTACK_RUN / 'input-watermarked.mp4'}",
    "--case", f"propainter={ATTACK_RUN / 'propainter.mp4'}",
    "--case", f"diffueraser={ATTACK_RUN / 'diffueraser.mp4'}",
])


In [ ]:
show_results(RUN_ROOT)
